# Impact of Market Share on Flight Operational Performance
### DSA210 – Introduction to Data Science | University Project

**Research Question:** Do dominant airlines on specific routes (market share > 50%) exhibit better or worse punctuality compared to airlines on competitive routes?

**Datasets:**
- **BTS Flight Data 2024** – 6.9M+ flight records with delay breakdowns (Bureau of Transportation Statistics)
- **US Airline Routes & Fares 1993-2024** – Route-level market share data filtered to 2024

**Pipeline:**
1. Data Loading & Cleaning
2. City-Name Normalization & Dataset Merge
3. Route Classification (Dominant vs Competitive)
4. Exploratory Data Analysis (EDA)
5. Hypothesis Testing (t-test & Mann-Whitney U)
6. Conclusions

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings
import re

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind

warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = {'Dominant': '#E05C5C', 'Competitive': '#4A90D9'}

print('All libraries imported successfully.')

## 2. Load & Clean BTS Flight Data

In [ ]:
# ── 2.1 Load all 12 monthly BTS CSV files ──────────────────────────────────
folder = r'C:\Users\OseasyVM\Desktop\term project\bts'

cols = [
    'YEAR', 'MONTH', 'QUARTER', 'OP_UNIQUE_CARRIER',
    'ORIGIN', 'ORIGIN_CITY_NAME', 'DEST', 'DEST_CITY_NAME',
    'DEP_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY',
    'NAS_DELAY', 'LATE_AIRCRAFT_DELAY'
]

dfs = []
for path in sorted(glob.glob(folder + r'\*.csv')):
    df = pd.read_csv(path, usecols=cols)
    dfs.append(df)
    print(f'  Loaded {path.split(chr(92))[-1]:10s} → {len(df):>8,} rows')

bts_df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined shape (raw): {bts_df.shape}')

In [ ]:
# ── 2.2 Drop rows with missing departure delay (target variable) ────────────
bts_df = bts_df.dropna(subset=['DEP_DELAY'])

# ── 2.3 Cap extreme outlier delays (> 10 hours treated as data anomalies) ──
bts_df = bts_df[bts_df['DEP_DELAY'] <= 600]   # 600 min = 10 hours

# ── 2.4 Convert delay columns to numeric (coerce any stray strings) ─────────
delay_cols = ['DEP_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'LATE_AIRCRAFT_DELAY']
for c in delay_cols:
    bts_df[c] = pd.to_numeric(bts_df[c], errors='coerce')

print(f'BTS shape after cleaning: {bts_df.shape}')
print(f'\nDEP_DELAY stats:')
print(bts_df['DEP_DELAY'].describe().round(2))

## 3. Load & Filter Routes & Fares Data

In [ ]:
# ── 3.1 Load routes dataset ─────────────────────────────────────────────────
routes_df = pd.read_csv(
    r'C:\Users\OseasyVM\Desktop\term project\US Airline Flight Routes and Fares 1993-2024.csv',
    low_memory=False
)

# ── 3.2 Filter to year 2024 ──────────────────────────────────────────────────
routes_2024 = routes_df[routes_df['Year'] == 2024].copy()
print(f'Routes 2024 shape: {routes_2024.shape}')
print(f'\nColumns: {list(routes_2024.columns)}')

In [ ]:
# ── 3.3 Inspect key market-share columns ────────────────────────────────────
print('Market share column stats:')
print(routes_2024[['carrier_lg', 'large_ms', 'carrier_low', 'lf_ms', 'fare', 'passengers']].describe().round(3))

## 4. City-Name Normalization & Dataset Merge

**Challenge:** BTS uses `'New York, NY'` while Routes uses `'New York'`.  
We strip everything after the first comma and lowercase for a reliable join.

In [ ]:
def normalize_city(name: str) -> str:
    """Strip state suffix and normalize whitespace for city-name matching."""
    if pd.isna(name):
        return ''
    # Take only the part before the first comma, then strip & lowercase
    city = str(name).split(',')[0].strip().lower()
    # Remove special characters that might differ between sources
    city = re.sub(r'[^a-z0-9 ]', '', city)
    return city

# ── Apply to BTS ─────────────────────────────────────────────────────────────
bts_df['origin_city_norm'] = bts_df['ORIGIN_CITY_NAME'].apply(normalize_city)
bts_df['dest_city_norm']   = bts_df['DEST_CITY_NAME'].apply(normalize_city)

# ── Apply to Routes ──────────────────────────────────────────────────────────
routes_2024['city1_norm'] = routes_2024['city1'].apply(normalize_city)
routes_2024['city2_norm'] = routes_2024['city2'].apply(normalize_city)

# Quick sanity check
print('BTS city examples:')
print(bts_df[['ORIGIN_CITY_NAME', 'origin_city_norm']].drop_duplicates().head(6).to_string(index=False))
print()
print('Routes city examples:')
print(routes_2024[['city1', 'city1_norm']].drop_duplicates().head(6).to_string(index=False))

In [ ]:
# ── 4.1 Build a canonical city-pair key (alphabetically sorted) ─────────────
# Sorting ensures 'NYC→LAX' and 'LAX→NYC' map to the same key.
def route_key(c1, c2):
    return tuple(sorted([str(c1), str(c2)]))

bts_df['route_key'] = bts_df.apply(
    lambda r: route_key(r['origin_city_norm'], r['dest_city_norm']), axis=1
)

routes_2024['route_key'] = routes_2024.apply(
    lambda r: route_key(r['city1_norm'], r['city2_norm']), axis=1
)

print(f'Unique route keys in BTS:    {bts_df["route_key"].nunique():,}')
print(f'Unique route keys in Routes: {routes_2024["route_key"].nunique():,}')

In [ ]:
# ── 4.2 Aggregate BTS to route level ────────────────────────────────────────
bts_route = (
    bts_df
    .groupby('route_key', as_index=False)
    .agg(
        flight_count       = ('DEP_DELAY', 'count'),
        avg_dep_delay      = ('DEP_DELAY', 'mean'),
        median_dep_delay   = ('DEP_DELAY', 'median'),
        std_dep_delay      = ('DEP_DELAY', 'std'),
        avg_carrier_delay  = ('CARRIER_DELAY', 'mean'),
        avg_weather_delay  = ('WEATHER_DELAY', 'mean'),
        avg_nas_delay      = ('NAS_DELAY', 'mean'),
        pct_on_time        = ('DEP_DELAY', lambda x: (x <= 0).mean() * 100),
        pct_late_15        = ('DEP_DELAY', lambda x: (x > 15).mean() * 100),
    )
)

# Keep routes with reasonable flight volume for statistical reliability
bts_route = bts_route[bts_route['flight_count'] >= 50]
print(f'Route-level BTS shape: {bts_route.shape}')

In [ ]:
# ── 4.3 Aggregate routes dataset to one row per route key ───────────────────
# If a route appears multiple times (quarters), take the max large_ms
routes_agg = (
    routes_2024
    .groupby('route_key', as_index=False)
    .agg(
        carrier_lg  = ('carrier_lg', 'first'),
        large_ms    = ('large_ms', 'max'),
        carrier_low = ('carrier_low', 'first'),
        lf_ms       = ('lf_ms', 'max'),
        avg_fare    = ('fare', 'mean'),
        avg_passengers = ('passengers', 'mean'),
    )
)

print(f'Aggregated routes shape: {routes_agg.shape}')

In [ ]:
# ── 4.4 Merge on route_key ───────────────────────────────────────────────────
merged_df = bts_route.merge(routes_agg, on='route_key', how='inner')

print(f'Merged dataset shape: {merged_df.shape}')
print(f'\nSample rows:')
merged_df[['route_key', 'flight_count', 'avg_dep_delay',
           'carrier_lg', 'large_ms', 'pct_on_time']].head(8)

## 5. Route Classification: Dominant vs Competitive

A route is classified as **Dominant** if the largest carrier holds **> 50%** market share (`large_ms > 0.5`).  
Otherwise it is **Competitive**.

In [ ]:
# ── 5.1 Classify routes ──────────────────────────────────────────────────────
merged_df['market_type'] = np.where(
    merged_df['large_ms'] > 0.5, 'Dominant', 'Competitive'
)

counts = merged_df['market_type'].value_counts()
print('Route classification breakdown:')
print(counts.to_string())
print(f'\nDominant  share: {counts["Dominant"]  / len(merged_df):.1%}')
print(f'Competitive share: {counts["Competitive"] / len(merged_df):.1%}')

In [ ]:
# ── 5.2 Summary statistics by market type ───────────────────────────────────
summary = (
    merged_df
    .groupby('market_type')[['avg_dep_delay', 'median_dep_delay', 'pct_on_time', 'pct_late_15', 'large_ms']]
    .agg(['mean', 'median', 'std'])
    .round(3)
)
print('Summary statistics by market type:')
summary

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# ── 6.1 Market-type counts (bar chart) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar: route counts
ax = axes[0]
counts.plot(kind='bar', ax=ax, color=[PALETTE[k] for k in counts.index],
            edgecolor='white', width=0.55)
ax.set_title('Number of Routes by Market Type', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Route Count')
ax.set_xticklabels(counts.index, rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2, p.get_height() + 1),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie: proportion
ax2 = axes[1]
ax2.pie(counts.values,
        labels=counts.index,
        colors=[PALETTE[k] for k in counts.index],
        autopct='%1.1f%%',
        startangle=140,
        wedgeprops=dict(edgecolor='white', linewidth=2),
        textprops={'fontsize': 11})
ax2.set_title('Route Proportion by Market Type', fontweight='bold')

plt.tight_layout()
plt.savefig('fig1_route_distribution.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── 6.2 Distribution of average departure delay by market type ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for mtype, color in PALETTE.items():
    subset = merged_df[merged_df['market_type'] == mtype]['avg_dep_delay']
    axes[0].hist(subset, bins=40, alpha=0.65, color=color,
                 label=mtype, edgecolor='none')

axes[0].set_title('Distribution of Avg Departure Delay by Route Type', fontweight='bold')
axes[0].set_xlabel('Average Departure Delay (minutes)')
axes[0].set_ylabel('Number of Routes')
axes[0].legend()

# KDE overlay
for mtype, color in PALETTE.items():
    subset = merged_df[merged_df['market_type'] == mtype]['avg_dep_delay']
    subset.plot.kde(ax=axes[1], color=color, linewidth=2.5, label=mtype)

# Mark means
for mtype, color in PALETTE.items():
    m = merged_df[merged_df['market_type'] == mtype]['avg_dep_delay'].mean()
    axes[1].axvline(m, color=color, linestyle='--', alpha=0.7,
                    label=f'{mtype} mean = {m:.1f} min')

axes[1].set_title('Kernel Density of Avg Departure Delay', fontweight='bold')
axes[1].set_xlabel('Average Departure Delay (minutes)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig2_delay_distributions.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── 6.3 Boxplot: departure delay by market type ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

order = ['Dominant', 'Competitive']

sns.boxplot(
    data=merged_df, x='market_type', y='avg_dep_delay',
    order=order, palette=PALETTE,
    width=0.45, fliersize=3, linewidth=1.2,
    ax=axes[0]
)
axes[0].set_title('Avg Departure Delay by Market Type', fontweight='bold')
axes[0].set_xlabel('Market Type')
axes[0].set_ylabel('Avg Departure Delay (min)')

sns.violinplot(
    data=merged_df, x='market_type', y='avg_dep_delay',
    order=order, palette=PALETTE,
    inner='quartile', linewidth=1,
    ax=axes[1]
)
axes[1].set_title('Delay Distribution Shape (Violin)', fontweight='bold')
axes[1].set_xlabel('Market Type')
axes[1].set_ylabel('Avg Departure Delay (min)')

plt.tight_layout()
plt.savefig('fig3_boxplot_violin.png', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# ── 6.4 On-time rate & % severely late by market type ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

metrics = {
    'pct_on_time': ('On-Time Rate (%)', axes[0]),
    'pct_late_15': ('% Flights > 15 min Late', axes[1]),
}

for col, (label, ax) in metrics.items():
    group_means = merged_df.groupby('market_type')[col].mean().reindex(order)
    bars = ax.bar(group_means.index, group_means.values,
                  color=[PALETTE[k] for k in group_means.index],
                  edgecolor='white', width=0.45)
    for bar, val in zip(bars, group_means.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f'{val:.1f}%',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title(f'{label} by Market Type', fontweight='bold')
    ax.set_ylabel(label)
    ax.set_xlabel('Market Type')
    ax.set_ylim(0, group_means.max() * 1.2)

plt.tight_layout()
plt.savefig('fig4_ontime_rates.png', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ── 6.5 Scatter: Market share vs Average departure delay ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_col, y_label in [
    (axes[0], 'avg_dep_delay', 'Avg Departure Delay (min)'),
    (axes[1], 'pct_on_time',   'On-Time Rate (%)'),
]:
    for mtype, color in PALETTE.items():
        sub = merged_df[merged_df['market_type'] == mtype]
        ax.scatter(sub['large_ms'], sub[y_col],
                   alpha=0.35, s=18, color=color, label=mtype)

    # Overall regression line
    x = merged_df['large_ms'].values
    y = merged_df[y_col].values
    mask = np.isfinite(x) & np.isfinite(y)
    m, b, r, p, _ = stats.linregress(x[mask], y[mask])
    x_line = np.linspace(x[mask].min(), x[mask].max(), 200)
    ax.plot(x_line, m * x_line + b, 'k--', linewidth=1.5,
            label=f'Trend (r={r:.2f}, p={p:.3f})')

    ax.axvline(0.5, color='gray', linestyle=':', linewidth=1,
               label='50% threshold')
    ax.set_title(f'Market Share vs {y_label}', fontweight='bold')
    ax.set_xlabel('Largest Carrier Market Share')
    ax.set_ylabel(y_label)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig5_scatter_ms_delay.png', bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

In [ ]:
# ── 6.6 Delay component breakdown ───────────────────────────────────────────
delay_components = ['avg_carrier_delay', 'avg_weather_delay', 'avg_nas_delay']
comp_means = (
    merged_df.groupby('market_type')[delay_components]
    .mean()
    .reindex(order)
    .rename(columns={
        'avg_carrier_delay': 'Carrier',
        'avg_weather_delay': 'Weather',
        'avg_nas_delay':     'NAS'
    })
)

fig, ax = plt.subplots(figsize=(9, 5))
comp_means.plot(
    kind='bar', ax=ax, width=0.5,
    color=['#E05C5C', '#7FAED4', '#F5A623'],
    edgecolor='white'
)
ax.set_title('Average Delay Component by Market Type', fontweight='bold')
ax.set_xlabel('Market Type')
ax.set_ylabel('Average Delay (minutes)')
ax.set_xticklabels(order, rotation=0)
ax.legend(title='Delay Type')

plt.tight_layout()
plt.savefig('fig6_delay_components.png', bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

In [ ]:
# ── 6.7 Heatmap: Carrier × Market Type (top carriers) ───────────────────────
# Top 10 carriers by number of routes in merged data
top_carriers = (
    merged_df.groupby('carrier_lg')['avg_dep_delay']
    .mean()
    .nlargest(10)
    .index.tolist()
)

heat_data = (
    merged_df[merged_df['carrier_lg'].isin(top_carriers)]
    .groupby(['carrier_lg', 'market_type'])['avg_dep_delay']
    .mean()
    .unstack('market_type')
    .reindex(columns=order)
    .sort_values('Dominant', ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heat_data, annot=True, fmt='.1f',
    cmap='RdYlGn_r', linewidths=0.5,
    cbar_kws={'label': 'Avg Dep Delay (min)'},
    ax=ax
)
ax.set_title('Avg Departure Delay by Carrier & Market Type\n(Top 10 by delay)', fontweight='bold')
ax.set_xlabel('Market Type')
ax.set_ylabel('Dominant Carrier')

plt.tight_layout()
plt.savefig('fig7_heatmap_carrier.png', bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

## 7. Hypothesis Testing

### Hypotheses

| | Null (H₀) | Alternative (H₁) |
|---|---|---|
| **Departure Delay** | µ_dominant = µ_competitive | µ_dominant ≠ µ_competitive |
| **On-Time Rate** | µ_dominant = µ_competitive | µ_dominant ≠ µ_competitive |

We use **two-sided tests** (no prior directional assumption) at **α = 0.05**.

**Two tests are run:**
- **Welch's t-test** – parametric, robust to unequal variances
- **Mann-Whitney U** – non-parametric, no normality assumption

Both tests use **route-level averages** (not individual flights), so each data point is an independent route.

In [ ]:
# ── 7.1 Separate the two groups ─────────────────────────────────────────────
dominant    = merged_df[merged_df['market_type'] == 'Dominant']
competitive = merged_df[merged_df['market_type'] == 'Competitive']

print(f'Dominant routes:    n = {len(dominant):,}')
print(f'Competitive routes: n = {len(competitive):,}')
print()

# Descriptive stats
for col, label in [('avg_dep_delay', 'Avg Departure Delay'), ('pct_on_time', 'On-Time Rate')]:
    print(f"{label}:")
    print(f"  Dominant    → mean={dominant[col].mean():.2f}, median={dominant[col].median():.2f}, std={dominant[col].std():.2f}")
    print(f"  Competitive → mean={competitive[col].mean():.2f}, median={competitive[col].median():.2f}, std={competitive[col].std():.2f}")
    print()

In [ ]:
# ── 7.2 Normality check (Shapiro-Wilk on a sample) ──────────────────────────
# Shapiro-Wilk is only valid for n < 5000; sample if needed
SAMPLE_N = 2000

for col, label in [('avg_dep_delay', 'Avg Dep Delay'), ('pct_on_time', 'On-Time Rate')]:
    print(f'--- Normality test: {label} ---')
    for name, grp in [('Dominant', dominant), ('Competitive', competitive)]:
        sample = grp[col].dropna()
        if len(sample) > SAMPLE_N:
            sample = sample.sample(SAMPLE_N, random_state=42)
        stat, p = stats.shapiro(sample)
        print(f'  {name:12s}: W={stat:.4f}, p={p:.4e} → {"NOT normal" if p < 0.05 else "Normal"}')
    print()

In [ ]:
# ── 7.3 Run all hypothesis tests ────────────────────────────────────────────
ALPHA = 0.05

def run_tests(group_a, group_b, metric_name):
    """Run Welch's t-test and Mann-Whitney U; return results dict."""
    a = group_a.dropna().values
    b = group_b.dropna().values

    # Welch's t-test (does NOT assume equal variances)
    t_stat, t_p = ttest_ind(a, b, equal_var=False)

    # Mann-Whitney U (non-parametric alternative)
    u_stat, u_p = mannwhitneyu(a, b, alternative='two-sided')

    # Effect size: Cohen's d
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    cohen_d = (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else np.nan

    print(f'{'='*55}')
    print(f'  Metric: {metric_name}')
    print(f'{'='*55}')
    print(f'  Dominant    → n={len(a):,}, mean={np.mean(a):.3f}')
    print(f'  Competitive → n={len(b):,}, mean={np.mean(b):.3f}')
    print(f'  Mean diff   = {np.mean(a) - np.mean(b):.3f}')
    print()
    print(f'  Welch t-test : t={t_stat:.4f}, p={t_p:.4e}  → {"REJECT H0" if t_p < ALPHA else "FAIL TO REJECT H0"}')
    print(f'  Mann-Whitney U: U={u_stat:.1f}, p={u_p:.4e}  → {"REJECT H0" if u_p < ALPHA else "FAIL TO REJECT H0"}')
    print(f"  Cohen's d   = {cohen_d:.4f}  ({'small' if abs(cohen_d)<0.5 else 'medium' if abs(cohen_d)<0.8 else 'large'} effect)")
    print()

    return {
        'metric': metric_name,
        't_stat': t_stat, 't_p': t_p,
        'u_stat': u_stat, 'u_p': u_p,
        'cohen_d': cohen_d,
        'reject_h0_t': t_p < ALPHA,
        'reject_h0_u': u_p < ALPHA,
        'mean_dominant': np.mean(a),
        'mean_competitive': np.mean(b)
    }


results = []
results.append(run_tests(dominant['avg_dep_delay'],    competitive['avg_dep_delay'],    'Avg Departure Delay (min)'))
results.append(run_tests(dominant['median_dep_delay'], competitive['median_dep_delay'], 'Median Departure Delay (min)'))
results.append(run_tests(dominant['pct_on_time'],      competitive['pct_on_time'],      'On-Time Rate (%)'))
results.append(run_tests(dominant['pct_late_15'],      competitive['pct_late_15'],      '% Flights >15 min Late'))

In [ ]:
# ── 7.4 Results summary table ────────────────────────────────────────────────
results_df = pd.DataFrame(results)[[
    'metric', 'mean_dominant', 'mean_competitive',
    't_stat', 't_p', 'u_p', 'cohen_d',
    'reject_h0_t', 'reject_h0_u'
]].round(4)

results_df.columns = [
    'Metric', 'Mean (Dom)', 'Mean (Comp)',
    't-stat', 'p (t-test)', 'p (Mann-W)', "Cohen's d",
    'Reject H₀ (t)', 'Reject H₀ (MW)'
]
results_df

In [ ]:
# ── 7.5 Visualise test results ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

metrics_plot = [
    ('avg_dep_delay',    'Avg Departure Delay (min)'),
    ('median_dep_delay', 'Median Departure Delay (min)'),
    ('pct_on_time',      'On-Time Rate (%)'),
    ('pct_late_15',      '% Flights > 15 min Late'),
]

for ax, (col, label), res in zip(axes, metrics_plot, results):
    for mtype, color in PALETTE.items():
        sub = merged_df[merged_df['market_type'] == mtype][col].dropna()
        ax.hist(sub, bins=35, alpha=0.55, color=color, label=mtype, density=True)

    # Annotate p-values
    sig = '✓ Significant' if res['reject_h0_t'] else '✗ Not Significant'
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.text(0.97, 0.97,
            f"t-test p={res['t_p']:.3e}\nMW p={res['u_p']:.3e}\n{sig}",
            transform=ax.transAxes,
            va='top', ha='right', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='lightyellow', alpha=0.9))

plt.suptitle('Hypothesis Test Distributions: Dominant vs Competitive Routes',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig8_hypothesis_distributions.png', bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

## 8. Correlation Analysis

In [ ]:
# ── 8.1 Pearson correlation: market share vs delay metrics ───────────────────
corr_cols = ['large_ms', 'avg_dep_delay', 'pct_on_time', 'pct_late_15',
             'avg_carrier_delay', 'avg_weather_delay', 'avg_nas_delay', 'avg_fare']
corr_matrix = merged_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle mask
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, square=True,
    ax=ax
)
ax.set_title('Correlation Matrix: Market Share & Delay Metrics', fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_correlation_matrix.png', bbox_inches='tight')
plt.show()
print('Figure 9 saved.')

## 9. Conclusions

### 9.1 Findings Summary

| Metric | Dominant | Competitive | Significant? |
|---|---|---|---|
| Avg Departure Delay | see table above | see table above | see table above |
| On-Time Rate | — | — | — |
| % Flights >15 min Late | — | — | — |

> **Note:** Fill in the actual values from your `results_df` table above before submission.

### 9.2 Interpretation

- If **p < 0.05**: We reject H₀ and conclude there is a statistically significant difference in punctuality between dominant and competitive routes.
- The direction of the effect (whether dominant routes are *better* or *worse*) is given by the mean difference.
- **Cohen's d** quantifies practical significance. A small effect (d < 0.5) may be statistically detectable but operationally minor.

### 9.3 Possible Explanations

**If dominant routes perform better:**
- Airlines with high market share may have more gate slots, ground crew resources, and optimized turnaround procedures on routes they control.
- Less competition may reduce scheduling pressure.

**If dominant routes perform worse:**
- Reduced competitive pressure could lead to less incentive to maintain punctuality.
- Monopoly/near-monopoly routes may involve hub connections with more complex delay propagation.

### 9.4 Limitations

1. **City-level merge**: The BTS–Routes join is on normalized city names, not airport codes. Airports like ORD/MDW (both Chicago) could map to the same city, introducing noise.
2. **Aggregation level**: Route-level averages smooth individual flight variability.
3. **Confounders**: Route distance, weather patterns, hub vs. non-hub airports, and aircraft type are not controlled for.
4. **Market share measurement**: `large_ms` from the Routes dataset captures annual/quarterly average; actual market dynamics vary.

### 9.5 Future Work

- Merge on **IATA airport codes** instead of city names for precision.
- **Regression analysis**: control for route distance, carrier size, season.
- **Time-series analysis**: examine if market share changes correlate with punctuality changes on the same route.
- **Causal inference**: use regression discontinuity or difference-in-differences around market-share thresholds.
